# DuckDB로 Parquet 구간 조회

파일 설정 → 스키마 확인 → 시계열 선택 → 구간 설정 → 조회 순서로 실행합니다.
먼저 저장소에서 `uv sync`를 실행하고 VS Code 커널을 `.venv/bin/python`으로 선택하세요.
샘플은 중첩 시계열이므로 목록을 펼칩니다. 일반 Parquet는 `SERIES_COLUMN = None`으로 설정하세요.

In [1]:
from pathlib import Path
from numbers import Integral

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# 저장소 또는 그 하위 폴더, GitHub 상위 폴더에서 실행할 수 있습니다.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "DuckDB_notebook"]
PROJECT_ROOT = next(
    (path for path in candidates
     if (path / "parquet_query.ipynb").is_file()
     and (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("DuckDB_notebook 폴더에서 노트북을 실행하세요.")
print(f"프로젝트: {PROJECT_ROOT}")
print(f"DuckDB: {duckdb.__version__}")

프로젝트: /home/bjeong/GitHub/DuckDB_notebook
DuckDB: 1.5.6


## 1. 파일 설정

읽으려는 parquet 파일 경로를 설정하세요. 여러 파일을 읽으려면 `samples/*.parquet`처럼 glob 패턴을 지정하세요.

In [ ]:
PARQUET_PATH = "samples/KR7005930003.parquet"
TIMEZONE = "Asia/Seoul"  # TIMESTAMPTZ 표시 및 시간대 없는 입력 해석에 사용

path = Path(PARQUET_PATH).expanduser()
parquet_path = str(path if path.is_absolute() else PROJECT_ROOT / path)

# 이 셀을 재실행할 때 이전 연결을 닫습니다.
if "con" in globals():
    con.close()
con = duckdb.connect(":memory:")
con.execute("SET TimeZone = ?", [TIMEZONE])
con.read_parquet(parquet_path, union_by_name=True).create_view("parquet_data")
raw_schema = con.execute("DESCRIBE parquet_data").fetchdf()
display(raw_schema[["column_name", "column_type"]])
print("원본 행 수:", con.execute("SELECT count(*) FROM parquet_data").fetchone()[0])

## 2. 조회 대상 선택

샘플의 `series_1m`, `series_5m`, `series_30m`은 STRUCT 목록입니다. 하나를 선택하면 각 항목이 한 행이 되고 `feature_id`가 함께 유지됩니다.
일반적인 평면 Parquet는 `SERIES_COLUMN = None`으로 설정하세요. 아래 스키마를 보고 조회 컬럼을 정합니다.

In [ ]:
SERIES_COLUMN = "series_1m"  # "series_5m", "series_30m", 또는 None


def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'


raw_types = dict(zip(raw_schema["column_name"], raw_schema["column_type"]))
if SERIES_COLUMN is None:
    con.execute("CREATE OR REPLACE TEMP VIEW query_data AS SELECT * FROM parquet_data")
else:
    if SERIES_COLUMN not in raw_types:
        raise ValueError(f"없는 시계열 컬럼: {SERIES_COLUMN}. 일반 표는 None을 지정하세요.")
    series_type = raw_types[SERIES_COLUMN]
    if not (series_type.startswith("STRUCT(") and series_type.endswith("[]")):
        raise ValueError("SERIES_COLUMN은 STRUCT 목록 컬럼이어야 합니다.")
    if "feature_id" not in raw_types:
        raise ValueError("중첩 시계열 모드에는 feature_id 컬럼이 필요합니다.")
    series_sql = quote_identifier(SERIES_COLUMN)
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW query_data AS
        SELECT feature_id, item.*
        FROM (
            SELECT feature_id, unnest({series_sql}) AS item
            FROM parquet_data
        )
    """)

schema = con.execute("DESCRIBE query_data").fetchdf()
column_types = dict(zip(schema["column_name"], schema["column_type"]))
display(schema[["column_name", "column_type"]])
display(con.execute("SELECT * FROM query_data LIMIT 5").fetchdf())

## 3. 구간 컬럼과 데이터 범위 확인

숫자, DATE, TIMESTAMP, TIMESTAMPTZ 컬럼을 지원합니다. 샘플의 `window_start_us`는 정수 그대로 사용합니다.
`*_us`라는 이름만으로 Unix epoch인지 하루 내 시각인지 판단하지 않습니다.
최솟값/최댓값은 선택한 시계열 전체 기준이며 NULL은 제외합니다.

In [ ]:
RANGE_COLUMN = "window_start_us"  # 일반 파일이면 실제 숫자/날짜 컬럼으로 수정

if RANGE_COLUMN not in column_types:
    raise ValueError(f"없는 구간 컬럼: {RANGE_COLUMN}. 위 스키마를 확인하세요.")
range_type = column_types[RANGE_COLUMN]
numeric_types = {
    "TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT",
    "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT", "UHUGEINT",
    "FLOAT", "DOUBLE", "REAL",
}
is_numeric = range_type in numeric_types or range_type.startswith("DECIMAL(")
is_temporal = range_type == "DATE" or range_type.startswith("TIMESTAMP")
if not (is_numeric or is_temporal):
    raise ValueError(f"구간 조회를 지원하지 않는 타입: {range_type}")
range_sql = quote_identifier(RANGE_COLUMN)
bounds = con.execute(f"""
    SELECT count(*) AS total_rows,
           count({range_sql}) AS non_null_rows,
           min({range_sql}) AS min_value,
           max({range_sql}) AS max_value
    FROM query_data
""").fetchdf()
display(bounds)
if "feature_id" in column_types:
    display(con.execute("""
        SELECT DISTINCT feature_id FROM query_data
        ORDER BY feature_id LIMIT 50
    """).fetchdf())

## 4. 원하는 구간 입력

**시작 포함, 종료 제외**: `START <= 값 < END`. 한쪽을 `None`으로 두면 해당 경계는 제한하지 않습니다.
기본값은 전체 범위 중 최대 1,000행입니다. NULL 구간값은 제외합니다.

- 숫자 컬럼 예: `START = 0`, `END = 60_000_000` (실제 범위를 먼저 확인하세요).
- 날짜/시간 컬럼 예: `START = "2026-11-01"`, `END = "2026-11-02"`.
- 특정 feature만 조회: `FEATURE_IDS = ["위에서 확인한 feature_id"]`.
- `SELECT_COLUMNS = None`이면 모든 컬럼을 반환합니다.

샘플에서는 항목의 시작값이 구간 안에 있는지를 조회합니다. 시간 구간 간 겹침 조회는 아닙니다.
같은 구간값의 행이 여러 개라면 `ORDER_COLUMNS`에 추가 정렬 기준을 지정할 수 있습니다.

In [ ]:
START = None
END = None
FEATURE_IDS = []  # 빈 목록은 모든 feature
SELECT_COLUMNS = None  # 예: ["feature_id", "window_start_us", "value_float64", "status"]
ORDER_COLUMNS = ["feature_id"] if "feature_id" in column_types else []
LIMIT = 1_000

## 5. 조회 실행

SQL에는 검증한 컬럼명만 넣고 입력값은 바인딩합니다. 조회 결과만 DataFrame으로 가져옵니다.
`LIMIT`은 반환 행 수의 제한이며, 원본 스캔이나 정렬 비용까지 제한하지 않습니다.

In [ ]:
def query_range(start, end, feature_ids, selected_columns, order_columns, limit):
    if isinstance(limit, bool) or not isinstance(limit, Integral) or limit <= 0:
        raise ValueError("LIMIT은 1 이상의 정수여야 합니다.")
    selected = list(column_types) if selected_columns is None else selected_columns
    if not isinstance(selected, (list, tuple)) or not selected:
        raise ValueError("SELECT_COLUMNS는 None 또는 비어 있지 않은 컬럼 목록이어야 합니다.")
    if not isinstance(order_columns, (list, tuple)):
        raise ValueError("ORDER_COLUMNS는 컬럼 목록이어야 합니다.")
    for name in [*selected, *order_columns]:
        if name not in column_types:
            raise ValueError(f"없는 컬럼: {name}")
    if not isinstance(feature_ids, (list, tuple)):
        raise ValueError("FEATURE_IDS는 목록이어야 합니다.")
    if feature_ids and "feature_id" not in column_types:
        raise ValueError("이 파일에는 feature_id 컬럼이 없습니다.")

    # 실제 컬럼 타입으로 먼저 변환하여 잘못된 경계값과 역순 구간을 검증합니다.
    # 날짜/시간도 문자열 비교가 아닌 DuckDB의 타입 비교를 사용합니다.
    cast_sql = f"CAST(? AS {range_type})"
    try:
        lower, upper = con.execute(
            f"SELECT {cast_sql}, {cast_sql}", [start, end]
        ).fetchone()
        if start is not None and end is not None:
            valid = con.execute(
                f"SELECT {cast_sql} < {cast_sql}", [start, end]
            ).fetchone()[0]
            if not valid:
                raise ValueError("START는 END보다 작아야 합니다.")
    except duckdb.Error as exc:
        raise ValueError(f"경계값을 {range_type} 타입으로 변환할 수 없습니다: {exc}") from exc

    predicates = [f"{range_sql} IS NOT NULL"]
    params = []
    if start is not None:
        predicates.append(f"{range_sql} >= {cast_sql}")
        params.append(start)
    if end is not None:
        predicates.append(f"{range_sql} < {cast_sql}")
        params.append(end)
    if feature_ids:
        placeholders = ", ".join("?" for _ in feature_ids)
        predicates.append(f'"feature_id" IN ({placeholders})')
        params.extend(feature_ids)
    projection = ", ".join(quote_identifier(name) for name in selected)
    ordering = list(dict.fromkeys([RANGE_COLUMN, *order_columns]))
    order_sql = ", ".join(quote_identifier(name) for name in ordering)
    sql = (
        f"SELECT {projection} FROM query_data "
        f"WHERE {' AND '.join(predicates)} ORDER BY {order_sql} LIMIT ?"
    )
    params.append(int(limit))
    return con.execute(sql, params).fetchdf(), sql, params


result, query_sql, query_parameters = query_range(
    START, END, FEATURE_IDS, SELECT_COLUMNS, ORDER_COLUMNS, LIMIT
)
print(f"조회 결과: {len(result):,}행 (최대 {LIMIT:,}행)")
if len(result) == LIMIT:
    print("LIMIT에 도달했습니다. 조건에 맞는 행이 더 있을 수 있습니다.")
if result.empty:
    print("조건에 맞는 행이 없습니다. 데이터 범위와 feature_id를 확인하세요.")
display(result)
print("실행 SQL:", query_sql)
print("바인딩 값:", query_parameters)

## 6. 결과 저장 (선택)

`EXPORT_CSV = True`일 때만 현재 `result`를 저장합니다. 전체 일치 데이터가 아니라 LIMIT가 적용된 결과입니다.
기존 파일은 덮어쓰지 않습니다. 내보낸 파일은 `exports/`에 저장되며 Git에서 제외됩니다.

In [ ]:
EXPORT_CSV = False
EXPORT_PATH = PROJECT_ROOT / "exports" / "query_result.csv"

if EXPORT_CSV:
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(EXPORT_PATH, index=False, encoding="utf-8-sig", mode="x")
    print(f"저장 완료: {EXPORT_PATH} ({len(result):,}행)")